<a href="https://colab.research.google.com/github/hursoo/big_k-modern_1/blob/main/gb_061_gb_feature_dtm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.개요

In [ ]:
# =================================================================
# 셀 1: 라이브러리 설치
# =================================================================
# 설치 과정 표시, 에러 출력은 숨김

!pip install -q tomotopy==0.13.0 numpy==1.23.5 kneed 2> /dev/null # 2> /dev/null : 에러 출력 메시지를 숨깁니다.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 12.4 MB/s eta 0:00:00


=> "런타임 / 세션 다시 시작"

In [ ]:
# =================================================================
# 셀 2: 기본 설정 및 라이브러리 임포트
# =================================================================
# "세션 다시 시작" 후에 이 셀부터 실행합니다.

# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')
print("\n✅ 구글 드라이브 마운트 완료!")

# 경로 지정
file_path = '/content/drive/MyDrive/big_km_history01/'

Mounted at /content/drive

✅ 구글 드라이브 마운트 완료!


In [ ]:
import sys
import os, re
import pandas as pd
import numpy as np
print(np.__version__)
import random
import warnings

from tomotopy import DMRModel
from tomotopy import TermWeight
from tomotopy import utils
import tomotopy as tp

import matplotlib.pyplot as plt

1.23.5


In [ ]:
from collections import Counter   # 글자 수 계산에 유용한 패키지

# 2.개벽의 특성 벡터 추출 1: 고빈도 단어
- 빈도 단위는 tfidf 값으로,
- 개수는 고빈도 단어 50개로 설정

In [ ]:
# 통합 데이터 불러오기
gb_df = pd.read_excel(file_path + 'result/gb_data_2(doc,1g2g,wn_cls).xlsx') ###
print(gb_df.shape)
gb_df.head(3)

(6802, 9)


,doc_id,doc_raw,doc_split_12gram,r_no,title,w_new,ho_no,grid_1,wn_cls
0,1,創刊辭 强者도 부르짖고 弱者도 부르짖으며 優者도 부르짖고 劣者도 부르짖도다 東西南北...,창간 辭 강자 약자 優者 劣者 동서 남북 사해 팔방 소리 소리 판단 좌우 間 다수 ...,1,創刊辭,uk01,1,01q,0
1,2,哲人은 말하되 多數 人民의 聲은 곳 神의 聲이라 하엿나니 神은 스스로 要求가 없는지...,哲人 다수 인민 요구 인민 소리 요구 발표 갈앙 인민 소리 갈앙 다수 인민 갈앙 요...,1,創刊辭,uk01,1,01q,0
2,3,世界를 알라 사람은 天使도 안이며 野獸도 안이오 오즉 사람일 뿐이로다 이만치 進化된...,세계 사람 야수 사람 진화 진화 지식 진화 도덕 동물 세계 천당 지옥 세계 진화 국...,2,世界를 알라,uk01,1,01q,0


In [ ]:
# 문서-단어 행렬

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def get_dtm(df, col_name, stopw, rank_n): # rank_n : 고빈도 단어 n 순위까지
    '''
    # 문서-단어 행렬(dtm) 산출 함수
    '''
    # 단어 종류 모두 벡터화. 1음절 이상
    tv = TfidfVectorizer(ngram_range=(1,1), stop_words=stopw) ## 두 글자 이상
    dtm = tv.fit_transform(df[col_name])

    # tfidf 합계를 사용해 고빈도 단어 추출 (희소 행렬로 작업)
    term_sums = np.array(dtm.sum(axis=0)).flatten()
    highword_indices = term_sums.argsort()[-rank_n:][::-1]  # 상위 rank_n 개 단어 인덱스
    highword_list = [tv.get_feature_names_out()[i] for i in highword_indices]

    # 고빈도 단어만 포함된 희소 행렬 생성
    feature_dtm = dtm[:, highword_indices]
    feature_df = pd.DataFrame(feature_dtm.toarray(), columns=highword_list, index=df.index)

    return feature_df

In [ ]:
# 함수 실행

stopw = ['문제', '금일', '관계']  # 제외할 단어
dtm_gb_df = get_dtm(gb_df, 'doc_split_12gram', stopw, 50)
dtm_gb_df

,사람,사회,朝鮮,생활,사상,운동,自己,세계,민족,主義,...,혁명,발달,현재,사업,意識,문명,목적,노동,생산,방법
0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.031663,0.00000,0.0,0.0,0.0,0.0,0.000000,0.077667,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.150628,0.07488,0.0,0.0,0.0,0.0,0.000000,0.277109,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.468555,0.00000,0.0,0.0,0.0,0.0,0.197656,0.670442,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.070227,0.00000,0.0,0.0,0.0,0.0,0.000000,0.430650,0.090315,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6797,0.068947,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6798,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6799,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6800,0.082986,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 단어 빈도 확인

dtm_gb_df.sum()

,0
사람,201.808757
사회,173.281553
朝鮮,152.025952
생활,142.297271
사상,131.171816
운동,109.253477
自己,108.402774
세계,107.769487
민족,105.871938
主義,103.643982


In [ ]:
hw50 = print(dtm_gb_df.columns.tolist())
hw50

['사람', '사회', '朝鮮', '생활', '사상', '운동', '自己', '세계', '민족', '主義', '계급', '시대', '경제', '朝鮮_人', '정신', '인류', '자유', '단체', '교육', '정치', '민중', '일본', '필요', '개인', '도덕', '자연', '국가', '인간', '의미', '理想', '종교', '중국', '문화', '현상', '일반', '조직', '생명', '역사', '농민', '過去', '혁명', '발달', '현재', '사업', '意識', '문명', '목적', '노동', '생산', '방법']


In [ ]:
dtm_gb_df.loc[6800,:].values

array([0.08298621, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.12241703, 0.        , 0.        , 0.        ,
       0.        , 0.12220194, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

# 3.개벽의 특성 벡터 추출 2: 토픽
- 토픽 8개로 설정
- 토픽 개수, 알파값, 베타값, seed 값은 다양하게 설정할 수 있으며, 그 과정은 이후 작업에서 본격적으로 살펴볼 것이다.
- 다만, 여기서는 '고빈도 단어' 특성 추출과 비교할 목적으로, 나중에 최종 확정된 토픽 결과물을 산출하는 데 사용한 설정값을 사용한다.

In [ ]:
# 1) OpenMP 스레드 수 제한 (결과 재현성을 위해)
os.environ["OMP_NUM_THREADS"] = "1"

# 2) 파이썬 자체 random, numpy 랜덤 시드 고정
def set_global_seeds(seed_value=1000):
    random.seed(seed_value)
    np.random.seed(seed_value)

# 3) `a_data`를 `metadata`로 변환하는 함수
def transform_a_data_to_metadata(misc: dict):
    return {'metadata': str(misc['a_data'])}

def run_dmr_model(gridL, lineL, num_topics=10, seed=100, iterations=1000, alpha=0.1, eta=0.01):
    """
    DMR 모델 실행 및 메타데이터 저장.
    """
    # 파이썬 랜덤, numpy 랜덤 시드 고정
    set_global_seeds(seed)

    print(f"\nTraining DMR Model with {num_topics} topics, alpha={alpha}, eta={eta}...")

    # DMR 모델 초기화 (tomotopy 내부 시드 설정)
    model = DMRModel(
        k=num_topics,
        seed=seed,
        tw=TermWeight.ONE,
        alpha=alpha,
        eta=eta
    )
    corpus = utils.Corpus()

    # 코퍼스에 문서 추가 (lineL은 이미 단어 리스트로 전처리되었다고 가정)
    for grid, tokens in zip(gridL, lineL):
        corpus.add_doc(tokens, a_data=grid)

    # 모델에 코퍼스 추가 (메타데이터 변환 포함)
    model.add_corpus(corpus, transform=transform_a_data_to_metadata)

    # 학습
    for i in range(0, iterations, 20):  # 20단위로 학습 반복
        model.train(20, workers=1)
        print(f"Iteration: {i + 20}\tLog-likelihood: {model.ll_per_word:.4f}")

    # --- ✨ 변경점: 토픽별 단어 수를 직접 계산 ---
    # DMRModel에는 count_by_topics 속성이 없으므로,
    # 모든 문서의 단어-토픽 할당을 기반으로 직접 계산합니다.
    topic_counts = [0] * model.k
    for doc in model.docs:
        for topic_idx in doc.topics:
            topic_counts[topic_idx] += 1

    print("\n<Topics>")
    # 각 토픽의 정보(단어 수, 상위 단어)를 가져와서 형식에 맞게 출력
    for i in range(model.k):
        # 직접 계산한 토픽별 단어 수 가져오기
        topic_word_count = topic_counts[i]

        # 토픽별 상위 단어 20개 가져오기
        top_words = model.get_topic_words(i, top_n=20)

        # 단어만 추출하여 공백으로 연결
        word_list = [word[0] for word in top_words]
        words_str = ' '.join(word_list)

        # 최종 형식으로 출력
        print(f"| #{i} ({topic_word_count}) : {words_str}")

    # 원래 코드와의 호환성을 위해 topics 리스트도 반환
    topics = [model.get_topic_words(i, top_n=20) for i in range(model.k)]

    return model, topics

In [ ]:
# --- 사용 예시 ---

# 1. 메타데이터 준비
gridL = gb_df['grid_1'].tolist()

# 2. 텍스트 데이터 준비 (각 문서를 단어 리스트로 변환)
lineL = gb_df['doc_split_12gram'].apply(lambda x: str(x).split()).tolist()

# 3. 모델 실행
model, topics = run_dmr_model(
    gridL,
    lineL,
    num_topics=8,
    seed=7, ####
    iterations=1000,
    alpha=0.05,
    eta=0.1
)


Training DMR Model with 8 topics, alpha=0.05, eta=0.1...
Iteration: 20	Log-likelihood: -7.8370
Iteration: 40	Log-likelihood: -7.7045
Iteration: 60	Log-likelihood: -7.6746
Iteration: 80	Log-likelihood: -7.6570
Iteration: 100	Log-likelihood: -7.6406
Iteration: 120	Log-likelihood: -7.6298
Iteration: 140	Log-likelihood: -7.6241
Iteration: 160	Log-likelihood: -7.6265
Iteration: 180	Log-likelihood: -7.6206
Iteration: 200	Log-likelihood: -7.6148
Iteration: 220	Log-likelihood: -7.6125
Iteration: 240	Log-likelihood: -7.6115
Iteration: 260	Log-likelihood: -7.6079
Iteration: 280	Log-likelihood: -7.6026
Iteration: 300	Log-likelihood: -7.6080
Iteration: 320	Log-likelihood: -7.6051
Iteration: 340	Log-likelihood: -7.6025
Iteration: 360	Log-likelihood: -7.6056
Iteration: 380	Log-likelihood: -7.6066
Iteration: 400	Log-likelihood: -7.6027
Iteration: 420	Log-likelihood: -7.6039
Iteration: 440	Log-likelihood: -7.6023
Iteration: 460	Log-likelihood: -7.6023
Iteration: 480	Log-likelihood: -7.6020
Iteration:

In [ ]:
# 문서별 토픽 구성 추출 및 DataFrame 생성 ---

# 토픽의 개수 확인
num_topics = model.k
topic_columns = [f'T{i}' for i in range(num_topics)]

# 결과를 저장할 리스트
doc_topic_distributions = []
document_indices = []

for i, doc in enumerate(model.docs):
    # doc.get_topic_dist()를 사용하여 해당 문서의 토픽 확률 분포를 가져옵니다.
    # 각 요소는 특정 토픽에 대한 문서의 확률입니다.
    topic_dist = doc.get_topic_dist()
    doc_topic_distributions.append(topic_dist)
    document_indices.append(f'{i}')

# NumPy 배열로 변환 후 DataFrame 생성
dt_df = pd.DataFrame(doc_topic_distributions, columns=topic_columns, index=document_indices)
dt_df

,T0,T1,T2,T3,T4,T5,T6,T7
0,0.100175,0.000584,0.005788,0.000916,0.699498,0.007016,0.181811,0.004213
1,0.242115,0.000204,0.016817,0.000319,0.022108,0.313191,0.403775,0.001470
2,0.013623,0.000518,0.005135,0.000812,0.357190,0.570561,0.048423,0.003738
3,0.170529,0.000538,0.005336,0.000844,0.058408,0.671052,0.089409,0.003884
4,0.701495,0.000366,0.003632,0.000575,0.066367,0.217282,0.007639,0.002644
...,...,...,...,...,...,...,...,...
6797,0.003296,0.014734,0.604653,0.085668,0.000684,0.001389,0.163286,0.126290
6798,0.004578,0.020463,0.062022,0.007873,0.000949,0.001929,0.449007,0.453179
6799,0.003746,0.016743,0.232563,0.051896,0.000777,0.001578,0.003735,0.688963
6800,0.004120,0.018417,0.005820,0.007086,0.000854,0.101735,0.004108,0.857859


# The End of Notes